In [2]:
pip install pyTelegramBotAPI

  Obtaining dependency information for pyTelegramBotAPI from https://files.pythonhosted.org/packages/7a/d5/fe2cf6873fee400eb24cd244b6c33eda724dfdcea3835cff0de97b018557/pytelegrambotapi-4.29.1-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/48.3 kB ? eta -:--:--
     ---------------- --------------------- 20.5/48.3 kB 640.0 kB/s eta 0:00:01
     ------------------------ ------------- 30.7/48.3 kB 435.7 kB/s eta 0:00:01
     -------------------------------- ----- 41.0/48.3 kB 487.6 kB/s eta 0:00:01
     -------------------------------------- 48.3/48.3 kB 304.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/294.8 kB ? eta -:--:--
   ----- --------------------------------- 41.0/294.8 kB 991.0 kB/s eta 0:00:01
   ---------------- ----------------------- 122.9/294.8 kB 1.4 MB/s eta 0:00:01
   ------------------------------ --------- 225.3/294.8 kB 1.7 MB/s eta 0:00:01
   -------------------------------------- - 286.7/294.8 kB 2.2 MB/s eta 0:00:01


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# import telebot

# TOKEN = "8544100639:AAG9YamM6DJeWM7AKQfnAIIoKGjQ61wfLwQ"
 
# bot = telebot.TeleBot(TOKEN)

In [16]:
# bot.polling(none_stop=True)

In [18]:
import telebot
bot = telebot.TeleBot(TOKEN)
 
# Обрабатываются все сообщения, содержащие команды '/start' or '/help'.
@bot.message_handler(commands=['start', 'help'])
def send_welcome(message):
    bot.send_message(message.chat.id, f"Welcome, {message.chat.username}")

bot.polling(none_stop=True)

In [24]:
import telebot
import requests
import json
from datetime import datetime

# --- КОНФИГУРАЦИЯ ---
TOKEN = "8544100639:AAG9YamM6DJeWM7AKQfnAIIoKGjQ61wfLwQ"  # Замените на токен вашего бота
CURRENCY_API_URL = "https://www.cbr-xml-daily.ru/daily_json.js"

SUPPORTED_CURRENCIES = {
    "USD": "Доллар США",
    "EUR": "Евро",
    "GBP": "Фунт стерлингов",
    "JPY": "Японская иена",
    "CNY": "Китайский юань",
    # Добавьте другие валюты по необходимости
}

# --- ИСКЛЮЧЕНИЕ ---
class APIException(Exception):
    """Пользовательское исключение для ошибок API и ввода пользователя."""
    pass

# --- КЛАСС ДЛЯ КОНВЕРТАЦИИ ВАЛЮТ ---
class CurrencyConverter:
    @staticmethod
    def get_price(base: str, quote: str, amount: float) -> tuple[float, str]:
        if base not in SUPPORTED_CURRENCIES:
            raise APIException(f"Валюта {base} не поддерживается.")
        if quote not in SUPPORTED_CURRENCIES:
            raise APIException(f"Валюта {quote} не поддерживается.")

        try:
            response = requests.get(CURRENCY_API_URL)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            raise APIException(f"Ошибка API: {e}")
        except json.JSONDecodeError as e:
            raise APIException(f"Ошибка JSON: {e}")

        raw_date = data["Date"]

        # Исправление формата даты
        if 'T' in raw_date and len(raw_date) > 19:
            base_part = raw_date[:19]
            offset_part = raw_date[19:]
            if offset_part and (offset_part[0].isdigit() or offset_part.startswith('0')):
                offset_part = '+' + offset_part
            fixed_date_str = base_part + offset_part
        else:
            fixed_date_str = raw_date

        try:
            date_obj = datetime.fromisoformat(fixed_date_str)
        except ValueError as e:
            raise APIException(f"Некорректная дата в API: {raw_date} ({e})")

        formatted_date = date_obj.strftime("%d.%m.%Y")

        valutes = data["Valute"]
        if base not in valutes:
            raise APIException(f"Курс {base} не найден.")
        if quote not in valutes:
            raise APIException(f"Курс {quote} не найден.")

        base_to_rub = valutes[base]["Value"]
        quote_to_rub = valutes[quote]["Value"]

        result = amount * base_to_rub / quote_to_rub
        return round(result, 2), formatted_date


# --- ИНИЦИАЛИЗАЦИЯ БОТА ---
bot = telebot.TeleBot(TOKEN)

# --- ОБРАБОТЧИКИ КОМАНД ---
@bot.message_handler(commands=["start", "help"])
def handle_start_help(message):
    text = (
        "Привет! Я бот для конвертации валют.\n\n"
        "Чтобы узнать цену, отправьте сообщение в формате:\n"
        "<валюта1> <валюта2> <количество>\n\n"
        "Примеры:\n"
        "USD EUR 100\n"
        "EUR RUB 50\n\n"
        "Доступные валюты:\n"
    )
    for code, name in SUPPORTED_CURRENCIES.items():
        text += f"{code} — {name}\n"
    text += "\nИспользуйте /values, чтобы увидеть список валют."
    bot.send_message(message.chat.id, text)

@bot.message_handler(commands=["values"])
def handle_values(message):
    text = "Поддерживаемые валюты:\n"
    for code, name in SUPPORTED_CURRENCIES.items():
        text += f"{code} — {name}\n"
    bot.send_message(message.chat.id, text)

# --- ОБРАБОТКА ТЕКСТОВЫХ СООБЩЕНИЙ ---
@bot.message_handler(content_types=["text"])
def handle_convert(message):
    try:
        # Разбиваем сообщение на части
        parts = message.text.strip().split()
        if len(parts) != 3:
            raise APIException("Неверный формат. Ожидается: <валюта1> <валюта2> <количество>")

        base, quote, amount_str = parts

        # Проверяем, что количество — число
        try:
            amount = float(amount_str)
        except ValueError:
            raise APIException("Количество должно быть числом.")

        # Получаем цену и дату курса
        result, rate_date = CurrencyConverter.get_price(base.upper(), quote.upper(), amount)

        # Формируем ответ
        text = (
            f"{amount} {base.upper()} = {result} {quote.upper()}\n"
            f"Курс актуален на: {rate_date}"
        )
        bot.send_message(message.chat.id, text)

    except APIException as e:
        bot.send_message(message.chat.id, f"Ошибка: {e}")
    except Exception as e:
        bot.send_message(message.chat.id, f"Неизвестная ошибка: {e}")

# --- ЗАПУСК БОТА ---
if __name__ == "__main__":
    print("Бот запущен...")
    bot.polling(none_stop=True)


Бот запущен...


In [20]:
# import telebot
# import requests
# import json

# # --- КОНФИГУРАЦИЯ ---
# TOKEN = "8544100639:AAG9YamM6DJeWM7AKQfnAIIoKGjQ61wfLwQ"  # Мой токен
# CURRENCY_API_URL = "https://www.cbr-xml-daily.ru/daily_json.js" # Смотрим сайт ЦБ РФ
# SUPPORTED_CURRENCIES = {
#     "USD": "Доллар США",
#     "EUR": "Евро",
#     "RUB": "Российский рубль",
#     # Добавьте другие валюты по необходимости
# }

# # --- ИСКЛЮЧЕНИЕ ---
# class APIException(Exception):
#     """Пользовательское исключение для ошибок API и ввода пользователя."""
#     pass

# # --- КЛАСС ДЛЯ КОНВЕРТАЦИИ ВАЛЮТ ---
# class CurrencyConverter:
#     @staticmethod
#     def get_price(base: str, quote: str, amount: float) -> float:
#         # Проверка поддержки валют
#         if base not in SUPPORTED_CURRENCIES:
#             raise APIException(f"Валюта {base} не поддерживается.")
#         if quote not in SUPPORTED_CURRENCIES:
#             raise APIException(f"Валюта {quote} не поддерживается.")

#         # Запрос к API ЦБ
#         try:
#             response = requests.get(CURRENCY_API_URL)
#             response.raise_for_status()
#             data = response.json()
#         except requests.exceptions.RequestException as e:
#             raise APIException(f"Ошибка при запросе к API ЦБ: {e}")
#         except json.JSONDecodeError as e:
#             raise APIException(f"Ошибка парсинга JSON: {e}")


#         # Получаем курсы из поля "Valute"
#         valutes = data["Valute"]

#         # Курс base к рублю
#         if base not in valutes:
#             raise APIException(f"Курс для валюты {base} не найден в данных ЦБ.")
#         base_to_rub = valutes[base]["Value"]

#         # Курс quote к рублю
#         if quote not in valutes:
#             raise APIException(f"Курс для валюты {quote} не найден в данных ЦБ.")
#         quote_to_rub = valutes[quote]["Value"]

#         # Конвертация: (base → RUB → quote)
#         # Формула: amount * (курс base/RUB) / (курс quote/RUB)
#         result = amount * base_to_rub / quote_to_rub
#         return round(result, 2)

# # --- ИНИЦИАЛИЗАЦИЯ БОТА ---
# bot = telebot.TeleBot(TOKEN)

# # --- ОБРАБОТЧИКИ КОМАНД ---
# @bot.message_handler(commands=["start", "help"])
# def handle_start_help(message):
#     text = (
#         "Привет! Я бот для конвертации валют.\n\n"
#         "Чтобы узнать цену, отправьте сообщение в формате:\n"
#         "<валюта1> <валюта2> <количество>\n\n"
#         "Примеры:\n"
#         "USD EUR 100\n"
#         "EUR RUB 50\n\n"
#         "Доступные валюты:\n"
#     )
#     for code, name in SUPPORTED_CURRENCIES.items():
#         text += f"{code} — {name}\n"
#     text += "\nИспользуйте /values, чтобы увидеть список валют."
#     bot.send_message(message.chat.id, text)

# @bot.message_handler(commands=["values"])
# def handle_values(message):
#     text = "Поддерживаемые валюты:\n"
#     for code, name in SUPPORTED_CURRENCIES.items():
#         text += f"{code} — {name}\n"
#     bot.send_message(message.chat.id, text)

# # --- ОБРАБОТКА ТЕКСТОВЫХ СООБЩЕНИЙ ---
# @bot.message_handler(content_types=["text"])
# def handle_convert(message):
#     try:
#         # Разбиваем сообщение на части
#         parts = message.text.strip().split()
#         if len(parts) != 3:
#             raise APIException("Неверный формат. Ожидается: <валюта1> <валюта2> <количество>")

#         base, quote, amount_str = parts

#         # Проверяем, что количество — число
#         try:
#             amount = float(amount_str)
#         except ValueError:
#             raise APIException("Количество должно быть числом.")

#         # Получаем цену
#         result = CurrencyConverter.get_price(base.upper(), quote.upper(), amount)

#         # Формируем ответ
#         text = f"{amount} {base.upper()} = {result} {quote.upper()}"
#         bot.send_message(message.chat.id, text)

#     except APIException as e:
#         bot.send_message(message.chat.id, f"Ошибка: {e}")
#     except Exception as e:
#         bot.send_message(message.chat.id, f"Неизвестная ошибка: {e}")

# # --- ЗАПУСК БОТА ---
# if __name__ == "__main__":
#     print("Бот запущен...")
#     bot.polling(none_stop=True)


Бот запущен...
